In [ ]:
import pandas as pd
import glob
import os

# Find the uploaded Excel file automatically
files = glob.glob("/content/*.xlsx")

print("Excel files found:")
for file in files:
    print(os.path.basename(file))

# Use the first Excel file found
file_path = files[0]

# Read the Orders sheet
df = pd.read_excel(file_path, sheet_name="Orders")

print("\nDataset loaded successfully!")
print("Shape:", df.shape)

# Display first 5 rows
display(df.head())

Excel files found:
Practice_olap Sample_dataset.xlsx

Dataset loaded successfully!
Shape: (9994, 21)


,Row ID,OrderID,OrderDate,ShipDate,ShipMode,CustomerID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11 00:00:00,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11 00:00:00,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16 00:00:00,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18 00:00:00,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18 00:00:00,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [ ]:
# Export the complete Orders dataset to CSV
df.to_csv("sales25_full.csv", index=False)

print("CSV created successfully!")
print("Rows:", len(df))

CSV created successfully!
Rows: 9994


In [ ]:
# Check the columns in our dataset
print("Columns in the dataset:\n")

for i, column in enumerate(df.columns, 1):
    print(f"{i}. {column}")

print("\nDataset information:")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Columns in the dataset:

1. Row ID
2. OrderID
3. OrderDate
4. ShipDate
5. ShipMode
6. CustomerID
7. Customer Name
8. Segment
9. Country
10. City
11. State
12. Postal Code
13. Region
14. Product ID
15. Category
16. Sub-Category
17. Product Name
18. Sales
19. Quantity
20. Discount
21. Profit

Dataset information:
Rows: 9994
Columns: 21


In [ ]:
# Convert OrderDate into proper date format
df["OrderDate"] = pd.to_datetime(df["OrderDate"])

# Create time hierarchy for OLAP operations
df["Year"] = df["OrderDate"].dt.year
df["Quarter"] = "Q" + df["OrderDate"].dt.quarter.astype(str)
df["Month"] = df["OrderDate"].dt.month_name()

# Display the date hierarchy
display(df[["OrderDate", "Year", "Quarter", "Month"]].head(10))

,OrderDate,Year,Quarter,Month
0,2016-11-08,2016,Q4,November
1,2016-11-08,2016,Q4,November
2,2016-06-12,2016,Q2,June
3,2015-10-11,2015,Q4,October
4,2015-10-11,2015,Q4,October
5,2014-06-09,2014,Q2,June
6,2014-06-09,2014,Q2,June
7,2014-06-09,2014,Q2,June
8,2014-06-09,2014,Q2,June
9,2014-06-09,2014,Q2,June


In [ ]:
# BASE CUBE
# Rows: Year -> Quarter -> Month
# Columns: Product Category
# Values: Sales and Profit

base_cube = pd.pivot_table(
    df,
    index=["Year", "Quarter", "Month"],
    columns="Category",
    values=["Sales", "Profit"],
    aggfunc="sum",
    fill_value=0
)

print("BASE CUBE")
display(base_cube)

BASE CUBE


Profit                                   Sales  \
Category                Furniture Office Supplies  Technology   Furniture   
Year Quarter Month                                                          
2014 Q1      February    120.6917        176.0910    565.5257   1839.6580   
             January     805.4665        788.9506    855.7736   6242.5250   
             March     -1128.6550       1270.4133    356.9716  14573.9560   
     Q2      April       554.3801       1000.2086   1934.2465   7944.8370   
             June        596.3153       3031.4326   1348.7765  13206.1256   
             May        -349.8776       1747.2044   1341.3828   6912.7870   
     Q3      August      663.4717       2767.6007   1887.0326   7320.3465   
             July        374.2467      -2482.0172   1266.2879  10821.0510   
             September  1858.5976       6699.5402   -230.0384  23816.4808   
     Q4      December   2011.8198       2064.3953   4907.3548  30645.9665   
             November   -297.8958       4411.9560   5178.0667  21564.8727   
             October     249.1645       1117.6406   2081.4522  12304.2470   
2015 Q1      February    394.8948       1814.1018    604.8542   3134.3740   
             January   -3014.2000        589.8969   -856.7039  11739.9416   
             March      1455.0573       5407.8482   2869.1923  12499.7830   
     Q2      April       338.6502       1477.3596   2371.4864  10475.6985   
             June       -412.6597       2195.3864   1552.8305   7714.1790   
             May         900.5847       1731.6139   2035.6704   9374.9505   
     Q3      August     -487.4709       2208.7531   3634.5262   9638.5922   
             July       -325.0938       1113.7829   2499.9592  13674.4200   
             September  1350.1185       2037.8837   4821.1605  26273.0230   
     Q4      December    428.8141       2631.9409   4956.2109  23085.8192   
             November   2992.0240       2800.0289   6682.7355  30880.8325   
             October    -605.5163       1090.9375   2331.9448  12026.6235   
2016 Q1      February    377.0352       1330.0009   3297.5434   3925.5510   
             January     303.9607       1604.4712    916.3914   7622.7430   
             March      -555.2680       1061.4056   3105.8304  12801.0920   
     Q2      April       364.8334      -1100.4821   3713.4636  13212.0900   
             June       -106.2665       2467.4894   2389.1552  13070.5720   
             May        1824.4739       2397.5024   4440.1701  15119.8350   
     Q3      August     -494.1457       1629.1913    927.0237  12483.2323   
             July       1038.9073       1123.9602   2270.0104  13068.5190   
             September   702.6025       3714.0417   4912.0134  27262.8809   
     Q4      December   2828.6715      11466.6686   3589.9692  36678.7150   
             November    819.8088       4686.9250  -1495.3263  31783.6288   
             October    -144.6600       4680.0550  11707.7475  11872.5770   
2017 Q1      February   -199.0421        953.8253    859.0888   6866.3374   
             January     -39.4402       3260.4467   3919.4326   5964.0320   
             March      1000.8532       2715.9624  11035.0759  10893.4448   
     Q2      April       202.4624       3370.6015  -2639.7739   9065.9581   
             June        904.9956       3378.9821   3939.3580  19008.5867   
             May         -72.8829       3153.7260   3261.7397  16957.5582   
     Q3      August      322.2390       6067.8582   2650.8585  15441.8740   
             July        324.6244       2563.2845   4064.7123  11813.0220   
             September  1548.6837       4847.7438   4595.1281  29028.2060   
     Q4      December   1146.7548       1774.3132   5562.2788  31407.4668   
             November    406.0637       3609.1029   5674.9371  37056.7150   
             October   -2526.9203       4040.7751   7761.4207  21884.0682   

                                                   
Category               Office Supplies Technolo

In [ ]:
# ROLL-UP OPERATION
# Roll up the time dimension from Month -> Quarter -> Year

rollup = pd.pivot_table(
    df,
    index="Year",
    columns="Category",
    values=["Sales", "Profit"],
    aggfunc="sum",
    fill_value=0
)

print("ROLL-UP: Year-wise Sales and Profit by Category")
display(rollup)

ROLL-UP: Year-wise Sales and Profit by Category


Profit                                    Sales                  \
Category  Furniture Office Supplies  Technology    Furniture Office Supplies   
Year                                                                           
2014      5457.7255      22593.4161  21492.8325  157192.8531      151776.412   
2015      3015.2029      25099.5338  33503.8670  170518.2370      137233.463   
2016      6959.9531      35061.2292  39773.9920  198901.4360      183939.982   
2017      3018.3913      39736.6217  50684.2566  215387.2692      246097.175   

                      
Category  Technology  
Year                  
2014      175278.233  
2015      162780.809  
2016      226364.180  
2017      271730.811

In [ ]:
# DRILL-DOWN OPERATION
# Drill down from Year -> Quarter -> Month

drilldown = pd.pivot_table(
    df,
    index=["Year", "Quarter", "Month"],
    columns="Category",
    values=["Sales", "Profit"],
    aggfunc="sum",
    fill_value=0
)

print("DRILL-DOWN: Monthly Sales and Profit by Quarter and Year")
display(drilldown)

DRILL-DOWN: Monthly Sales and Profit by Quarter and Year


Profit                                   Sales  \
Category                Furniture Office Supplies  Technology   Furniture   
Year Quarter Month                                                          
2014 Q1      February    120.6917        176.0910    565.5257   1839.6580   
             January     805.4665        788.9506    855.7736   6242.5250   
             March     -1128.6550       1270.4133    356.9716  14573.9560   
     Q2      April       554.3801       1000.2086   1934.2465   7944.8370   
             June        596.3153       3031.4326   1348.7765  13206.1256   
             May        -349.8776       1747.2044   1341.3828   6912.7870   
     Q3      August      663.4717       2767.6007   1887.0326   7320.3465   
             July        374.2467      -2482.0172   1266.2879  10821.0510   
             September  1858.5976       6699.5402   -230.0384  23816.4808   
     Q4      December   2011.8198       2064.3953   4907.3548  30645.9665   
             November   -297.8958       4411.9560   5178.0667  21564.8727   
             October     249.1645       1117.6406   2081.4522  12304.2470   
2015 Q1      February    394.8948       1814.1018    604.8542   3134.3740   
             January   -3014.2000        589.8969   -856.7039  11739.9416   
             March      1455.0573       5407.8482   2869.1923  12499.7830   
     Q2      April       338.6502       1477.3596   2371.4864  10475.6985   
             June       -412.6597       2195.3864   1552.8305   7714.1790   
             May         900.5847       1731.6139   2035.6704   9374.9505   
     Q3      August     -487.4709       2208.7531   3634.5262   9638.5922   
             July       -325.0938       1113.7829   2499.9592  13674.4200   
             September  1350.1185       2037.8837   4821.1605  26273.0230   
     Q4      December    428.8141       2631.9409   4956.2109  23085.8192   
             November   2992.0240       2800.0289   6682.7355  30880.8325   
             October    -605.5163       1090.9375   2331.9448  12026.6235   
2016 Q1      February    377.0352       1330.0009   3297.5434   3925.5510   
             January     303.9607       1604.4712    916.3914   7622.7430   
             March      -555.2680       1061.4056   3105.8304  12801.0920   
     Q2      April       364.8334      -1100.4821   3713.4636  13212.0900   
             June       -106.2665       2467.4894   2389.1552  13070.5720   
             May        1824.4739       2397.5024   4440.1701  15119.8350   
     Q3      August     -494.1457       1629.1913    927.0237  12483.2323   
             July       1038.9073       1123.9602   2270.0104  13068.5190   
             September   702.6025       3714.0417   4912.0134  27262.8809   
     Q4      December   2828.6715      11466.6686   3589.9692  36678.7150   
             November    819.8088       4686.9250  -1495.3263  31783.6288   
             October    -144.6600       4680.0550  11707.7475  11872.5770   
2017 Q1      February   -199.0421        953.8253    859.0888   6866.3374   
             January     -39.4402       3260.4467   3919.4326   5964.0320   
             March      1000.8532       2715.9624  11035.0759  10893.4448   
     Q2      April       202.4624       3370.6015  -2639.7739   9065.9581   
             June        904.9956       3378.9821   3939.3580  19008.5867   
             May         -72.8829       3153.7260   3261.7397  16957.5582   
     Q3      August      322.2390       6067.8582   2650.8585  15441.8740   
             July        324.6244       2563.2845   4064.7123  11813.0220   
             September  1548.6837       4847.7438   4595.1281  29028.2060   
     Q4      December   1146.7548       1774.3132   5562.2788  31407.4668   
             November    406.0637       3609.1029   5674.9371  37056.7150   
             October   -2526.9203       4040.7751   7761.4207  21884.0682   

                                                   
Category               Office Supplies Technolo

In [ ]:
# SLICE OPERATION
# Select one Region: West

slice_west = df[df["Region"] == "West"]

slice_result = pd.pivot_table(
    slice_west,
    index=["Year", "Quarter", "Month"],
    columns="Category",
    values=["Sales", "Profit"],
    aggfunc="sum",
    fill_value=0
)

print("SLICE: Region = West")
display(slice_result)

SLICE: Region = West


Profit                                  Sales  \
Category                Furniture Office Supplies Technology   Furniture   
Year Quarter Month                                                         
2014 Q1      February    -18.7612        157.7806    86.3892    144.1280   
             January    -324.1080        348.5009     0.0000   1158.4350   
             March       161.5101        862.2130   510.5042   3920.7320   
     Q2      April       490.6007        328.6456   810.2082   3531.1010   
             June        337.1522       1186.9953   247.6677   3425.5480   
             May          95.9512        324.8319   452.7809   1864.3820   
     Q3      August        2.9096       1792.3631   765.6317   1600.7000   
             July        346.4137       1047.0329   553.2362   6792.2710   
             September   872.1845       1034.8221  1168.5161   7086.8540   
     Q4      December    623.4018        404.4459   853.2949  11461.7925   
             November    216.7305       1872.5636  1062.0166   6038.6385   
             October     258.0432        614.2724   518.9499   3057.8660   
2015 Q1      February     63.1284        309.7526    80.8380    376.3840   
             January     205.9154        178.3952   215.9973   3635.3140   
             March      1186.8417        816.6615  2220.3145   6559.9480   
     Q2      April       -15.2675        670.6496   677.4136   2316.1520   
             June        296.2064        161.1291   577.0106   1932.4900   
             May         428.3564        669.3959   543.3981   3752.4655   
     Q3      August       -3.6137        865.7029   376.8811   4598.8970   
             July       -417.4947        133.5912   308.1531   4915.6230   
             September   753.2366       1150.3537   547.6273   5837.9490   
     Q4      December    273.7238       1632.3467  2056.9164  11835.1290   
             November    346.5902        922.1919   481.4039   6452.3850   
             October     -32.5760       1158.5765   652.4470   4792.2070   
2016 Q1      February     10.9228        247.9884   129.4869    572.0960   
             January     119.2215        231.0282   293.9957   1925.4640   
             March        74.6354        355.3524   780.2307   2622.9100   
     Q2      April      -506.6712        440.4533   422.9760   6917.3740   
             June        337.0086        780.7740  1157.6737   6621.2470   
             May         626.0698        321.5384   536.6933   4920.5460   
     Q3      August      350.4686        442.3997   489.1668   6291.0620   
             July        597.3165        632.0807  1716.9442   5818.0130   
             September   413.9020       1281.8822  2580.6200   7720.0045   
     Q4      December     57.8564       2187.3507  1082.0727  14391.7520   
             November    965.8707       2580.1186   896.6356  11148.7460   
             October    -392.6698       2344.4597  -534.2463   4870.4260   
2017 Q1      February    150.0206        848.0962   650.1131   3099.3070   
             January     290.6769       1203.9571  1713.6989   2038.5060   
             March       128.5815       1219.8984  7766.5654   1674.3340   
     Q2      April       266.2144        896.9505 -2910.6197   2294.6780   
             June        592.3337       1298.4366   888.7471   6533.2815   
             May        -437.0459        879.3889  2208.3973   2921.5680   
     Q3      August      641.7210       4220.3045  1283.5003   4646.8160   
             July        224.1663        826.9948  3177.5937   4686.8630   
             September   527.7572       3515.0448   841.7798  11691.0480   
     Q4      December      1.8532       2141.6239  2025.8478  12362.4310   
             November    780.1552       1571.0737  1092.0208  14154.6790   
             October    -462.4904       3499.4392   246.1593   5602.2000   

                                                   
Category               Office Supplies Technology  
Year Quarter Month                            

In [ ]:
# DICE OPERATION
# Select multiple values from multiple dimensions

dice_data = df[
    (df["Region"].isin(["West", "East"])) &
    (df["Year"].isin([2014, 2015])) &
    (df["Category"].isin(["Furniture", "Technology"]))
]

dice_result = pd.pivot_table(
    dice_data,
    index=["Year", "Region"],
    columns="Category",
    values=["Sales", "Profit"],
    aggfunc="sum",
    fill_value=0
)

print("DICE: East & West | 2014 & 2015 | Furniture & Technology")
display(dice_result)

DICE: East & West | 2014 & 2015 | Furniture & Technology


Profit                   Sales           
Category     Furniture  Technology   Furniture Technology
Year Region                                              
2014 East    -499.6581  10833.9413  47232.7390  45478.729
     West    3062.0283   7029.1956  50082.4480  44953.731
2015 East     -47.4132  10097.7838  53817.4320  59859.380
     West    3085.0470   8738.4009  57004.9435  45097.774